# Practical 4 — Anomaly Detection Using an Autoencoder

Train an autoencoder on normal synthetic signals and detect unusual signals using reconstruction error.

## Aim
Demonstrate anomaly detection using an autoencoder trained on normal synthetic signals.

## Important note
This is a self-contained teaching example. It creates synthetic periodic signals rather than downloading a medical or financial dataset. Anomalies are artificial spikes. The purpose is to demonstrate reconstruction error and threshold-based detection.

In [ ]:
# 1. Import libraries
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense

In [ ]:
# 2. Create reproducible synthetic signals
rng = np.random.default_rng(42)
signal_length = 50

def make_normal_signal():
    t = np.linspace(0, 2 * np.pi, signal_length)
    phase = rng.uniform(-0.15, 0.15)
    amplitude = rng.uniform(0.9, 1.1)
    noise = rng.normal(0, 0.05, signal_length)
    return amplitude * np.sin(t + phase) + noise

def make_anomalous_signal():
    signal = make_normal_signal()
    spike_index = rng.integers(5, signal_length - 5)
    signal[spike_index] += rng.choice([-1.8, 1.8])
    return signal

normal_signals = np.array([make_normal_signal() for _ in range(1800)], dtype="float32")
x_train = normal_signals[:1400]
x_test_normal = normal_signals[1400:1600]
x_test_anomaly = np.array([make_anomalous_signal() for _ in range(200)], dtype="float32")

x_test = np.concatenate([x_test_normal, x_test_anomaly], axis=0)
y_test = np.concatenate([
    np.zeros(len(x_test_normal), dtype=int),
    np.ones(len(x_test_anomaly), dtype=int)
])

print("Training normal signals:", x_train.shape)
print("Test signals:", x_test.shape)

In [ ]:
# 3. Visualize normal and anomalous examples
plt.figure(figsize=(10, 4))
plt.plot(x_train[0], label="Normal example")
plt.plot(x_test_anomaly[0], label="Anomalous example")
plt.title("Synthetic Signal Examples")
plt.xlabel("Time step")
plt.ylabel("Signal value")
plt.legend()
plt.show()

In [ ]:
# 4. Build an autoencoder
model = Sequential([
    Input(shape=(signal_length,)),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),   # compressed representation
    Dense(32, activation="relu"),
    Dense(signal_length, activation="linear")
])

# 5. Compile for reconstruction
model.compile(optimizer="adam", loss="mse")
model.summary()

In [ ]:
# 6. Train only on normal signals
history = model.fit(
    x_train, x_train,
    epochs=15,
    batch_size=32,
    validation_split=0.1,
    shuffle=True,
    verbose=1
)

In [ ]:
# 7. Set anomaly threshold from training reconstruction errors
train_reconstruction = model.predict(x_train, verbose=0)
train_errors = np.mean(np.square(x_train - train_reconstruction), axis=1)
threshold = np.percentile(train_errors, 95)

# 8. Reconstruct test signals and calculate their errors
test_reconstruction = model.predict(x_test, verbose=0)
test_errors = np.mean(np.square(x_test - test_reconstruction), axis=1)
y_pred = (test_errors > threshold).astype(int)

print("Anomaly threshold:", round(float(threshold), 5))
print("Predicted anomalies:", int(y_pred.sum()), "out of", len(y_pred))
print("Actual anomalies:", int(y_test.sum()))

In [ ]:
# 9. Plot training loss
plt.figure(figsize=(6, 4))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("Autoencoder Reconstruction Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.show()

# 10. Plot test reconstruction errors and threshold
plt.figure(figsize=(10, 4))
plt.plot(test_errors, marker=".", linestyle="none", label="Test reconstruction error")
plt.axhline(threshold, linestyle="--", label="Anomaly threshold")
plt.title("Reconstruction Error-Based Anomaly Detection")
plt.xlabel("Test sample index")
plt.ylabel("Mean squared reconstruction error")
plt.legend()
plt.show()

In [ ]:
# 11. Show a few predictions
for i in [0, 50, 199, 200, 250, 350]:
    label_text = "ANOMALY" if y_pred[i] == 1 else "NORMAL"
    print(
        f"Sample {i:3d} | actual={'ANOMALY' if y_test[i] else 'NORMAL':8s} "
        f"| predicted={label_text:8s} | error={test_errors[i]:.5f}"
    )

## Result
An autoencoder is trained on normal synthetic signals. Reconstruction errors are calculated, a percentile-based threshold is selected, and test signals are labelled as normal or anomalous.